# Enums

**Plain language:** An enum is a way to give a fixed, named set of choices a real identity in your code, instead of representing them as raw strings or magic numbers.

**Analogy:** Think of an HTTP status code. You *could* pass around the raw integer `404` everywhere, but that's error-prone — nothing stops you from typing `400` by mistake, and the number alone doesn't tell you what it means. An enum is like giving every status code a label (`NOT_FOUND`) so the code is self-documenting and the set of valid values is locked down.

**Where you've already used this idea without naming it:** Django's `TextChoices` / `IntegerChoices` on model fields (e.g. `Order.Status.PENDING`) are Python enums under the hood.

In [5]:
import enum


class VideoStatus(enum.StrEnum):
    """Lifecycle states for an HLS video transcode job."""

    PENDING = "pending"
    PROCESSING = "processing"
    READY = "ready"
    FAILED = "failed"


lv = lambda x: [e.value for e in x]
ln = lambda x: [e.name for e in x]

In [7]:
result_values = lv(VideoStatus)
result_names = ln(VideoStatus)

In [8]:
result_values, result_names

(['pending', 'processing', 'ready', 'failed'],
 ['PENDING', 'PROCESSING', 'READY', 'FAILED'])

## Minimal example

In [10]:
from enum import Enum


class OrderStatus(Enum):
    PENDING = "pending"
    PAID = "paid"
    SHIPPED = "shipped"
    CANCELLED = "cancelled"


status = OrderStatus.PAID
print(status)            # OrderStatus.PAID
print(status.name)       # "PAID"
print(status.value)      # "paid"
print(status is OrderStatus.PAID)  # True — members are singletons

OrderStatus.PAID
PAID
paid
True


## Iterating over members

An `Enum` class implements the iterator protocol itself — you can loop over it directly, in *definition order*, and it yields each member one at a time.

In [12]:
for member in OrderStatus:
    print(member.name, "->", member.value)

PENDING -> pending
PAID -> paid
SHIPPED -> shipped
CANCELLED -> cancelled


That loop works because `Enum` defines `__iter__` on the *metaclass* (`EnumMeta`), which yields members in the order they were declared — not alphabetically, not by value.

You can get the same sequence as a list via `list(OrderStatus)`, and `len(OrderStatus)` tells you how many members exist.

In [14]:
print(list(OrderStatus))
print(len(OrderStatus))

[<OrderStatus.PENDING: 'pending'>, <OrderStatus.PAID: 'paid'>, <OrderStatus.SHIPPED: 'shipped'>, <OrderStatus.CANCELLED: 'cancelled'>]
4


In [15]:
# Lookup by value or by name
print(OrderStatus("paid"))        # OrderStatus.PAID  — lookup by value
print(OrderStatus["PAID"])        # OrderStatus.PAID  — lookup by name

OrderStatus.PAID
OrderStatus.PAID


## Gotchas

- **Aliases don't show up twice when iterating.** If two names point to the same value, the second one becomes an *alias* for the first, and iteration skips it.
- **`auto()`** can generate values for you when you don't care what the underlying value is, only that each member is distinct.
- **`IntEnum` / `StrEnum`** (3.11+) let members behave like `int`/`str` directly (`==` comparisons against plain ints/strings work), which `Enum` alone does not allow.

In [4]:
from enum import Enum, auto


class Direction(Enum):
    NORTH = auto()
    SOUTH = auto()
    EAST = auto()
    WEST = auto()
    # Alias — same value as NORTH, will NOT appear when iterating
    UP = NORTH


for d in Direction:
    print(d.name, d.value)

print(Direction.UP is Direction.NORTH)  # True — alias, not a separate member

NORTH 1
SOUTH 2
EAST 3
WEST 4
True
